# Retrieval-Augmented Generation Pipeline

Run a question through the new financial-report pipeline: retrieve candidates, rerank evidence, then ask the chat model for a cited answer. Run from this repository in the configured `assignment` kernel.

In [1]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'retrieval' / 'search.py').exists())
sys.path.insert(0, str(project_root))

from retrieval.search import retrieve
from reranking.build import rerank
from generation.answer import generate

## Configure one query

Point `INDEX_MANIFEST_PATH` to a completed Milvus index manifest produced by `indexing.build`. Set `MILVUS_URI` and, if required, `MILVUS_TOKEN`. For generation, set `RAG_CHAT_MODEL` plus `RAG_CHAT_BASE_URL` for the course chat endpoint and `RAG_CHAT_API_KEY` if it requires a key. You can also use `OPENAI_API_KEY` for OpenAI.

In [ ]:
index_manifest_path = Path(os.environ.get('INDEX_MANIFEST_PATH', 'PATH/TO/index_manifest.json'))
output_dir = project_root / 'artifacts' / 'rag_queries'
question = 'What was the financial year end reported by the company?'
document_id = None  # Set to a document ID to restrict search.
page = None  # Set to a page number to restrict search.

if not index_manifest_path.is_file():
    raise FileNotFoundError('Set INDEX_MANIFEST_PATH to a completed Milvus index manifest')
print('Question:', question)
print('Index manifest:', index_manifest_path)

## Phase 1: Retrieve and inspect evidence

The retriever queries every path supported by the chosen index and fuses ranks. The reranker then selects the passages that will be sent to the model.

In [ ]:
retrieval_result = retrieve(index_manifest_path, question, output_dir / 'retrieval',
                            document_id=document_id, page=page,
                            path_limit=20, candidate_limit=20)
print('Retrieval trace:', retrieval_result['trace_path'])
for item in retrieval_result['fused_candidates'][:5]:
    print(item['fused_rank'], item['chunk_id'], item['page_range'], round(item['rrf_score'], 4))

In [ ]:
reranking_result = rerank(retrieval_result['trace_path'], output_dir / 'reranking',
                          max_candidates=20, evidence_limit=5)
print('Reranking trace:', reranking_result['reranking_trace_path'])
for number, item in enumerate(reranking_result['selected_evidence'], 1):
    print(f"[{number}] {item['document_id']} pages {item['page_range']}: {item['text'][:220]}...")

## Phase 2: Answer with citations

The answer cites numbered evidence passages. The saved answer JSON also keeps the source document, pages and chunk IDs for each citation.

In [ ]:
answer_result = generate(reranking_result['reranking_trace_path'],
                         output_dir / 'generation')
print(answer_result['answer'])
print('Citations:')
for citation in answer_result['citations']:
    print(f"[{citation['number']}] {citation['document_id']} pages {citation['page_range']} chunk {citation['chunk_id']}")
print('Answer trace:', answer_result['answer_path'])